# 01 — Data Quality Checks & Conversion by Group (SQL)
Prepares a clean table in DuckDB, runs data quality and sample ratio
checks, and computes the conversion rate per group.

In [2]:
import os
from pathlib import Path
import duckdb

if Path.cwd().name == "notebooks":
    os.chdir("..")
    
Path("data/processed").mkdir(parents=True, exist_ok=True)

con = duckdb.connect("data/processed/ab_test.duckdb")

def run_sql_file(path):
    """Jalankan semua query di file .sql dan tampilkan hasilnya satu per satu."""
    sql_text = Path(path).read_text()
    for query in sql_text.split(";"):
        if query.strip():
            result = con.sql(query)
            if result is not None:
                display(result.df())


In [3]:
run_sql_file("sql/01_create_clean_table.sql")

,Count
0,588101


In [4]:
run_sql_file("sql/02_data_quality.sql")

,n_rows,n_unique_users
0,588101,588101


,n_users_in_both_groups
0,0


,null_user_id,null_test_group,null_converted,null_total_ads,null_most_ads_day,null_most_ads_hour
0,0,0,0,0,0,0


,min_total_ads,max_total_ads,min_hour,max_hour
0,1,2065,0,23


,most_ads_day,n_users
0,Friday,92608
1,Monday,87073
2,Sunday,85391
3,Thursday,82982
4,Saturday,81660
5,Wednesday,80908
6,Tuesday,77479


,test_group,n_users,pct_users
0,ad,564577,96.0
1,psa,23524,4.0


In [5]:
from scipy.stats import chisquare

n_ad, n_psa = 564577, 23524
n_total = n_ad + n_psa

# Uji SRM dengan ASUMSI desain 96/4
stat, p_srm = chisquare([n_ad, n_psa], f_exp=[n_total * 0.96, n_total * 0.04])
print(f"SRM chi-square p-value (assumed 96/4 design): {p_srm:.4f}")

SRM chi-square p-value (assumed 96/4 design): 0.9998


### Data quality summary
- 588,101 rows, one row per unique user — no duplicates
- No user appears in both groups
- No missing values in any column
- `most_ads_hour` ranges from 0 to 23; all 7 days are present,
  with a fairly even distribution (77K–93K users per day)
- Group allocation: ad 96.0% (564,577 users) / psa 4.0% (23,524 users)
- The planned allocation ratio is not documented, but the observed
  split matches a 96/4 design almost exactly. An SRM chi-square test
  against an assumed 96/4 design gives p ≈ 1.00, so there is no sign
  of a sample ratio mismatch.

In [6]:
run_sql_file("sql/03_conversion_by_group.sql")

,test_group,n_users,n_converted,conversion_rate_pct
0,ad,564577,14423,2.555
1,psa,23524,420,1.785


### Conversion rate by group
- ad: 2.56%
- psa: 1.79%

The ad group converts about 0.77 percentage points higher than
the control group (≈43% relative lift). Whether this difference
is statistically significant is tested in the next notebook.

In [7]:
con.close()